# VitalGraph — Health Intelligence Pipeline

**Datasets:**
- `diabetes_dataset.csv` — 100K patient records (vitals, labs, lifestyle, diabetes staging)
- `activity_summary.xlsx` — 138 device benchmark rows (Apple Watch, Garmin, Fitbit signal quality)
- `Wearable_Dataset/` — PhysioNet E4 wristband data (STRESS / AEROBIC / ANAEROBIC sessions)

**Pipeline:**
1. Diabetes ML: XGBoost classifier + risk regressor + K-Means archetypes + Isolation Forest
2. Device Benchmark Layer: activity_summary signal quality analysis
3. Neo4j Graph: Patient → RiskProfile → SIMILAR_TO edges from clusters
4. PhysioNet Wearable Layer: HR/EDA/TEMP feature extraction + stress classifier
5. LLM Recommendations: Claude API health summaries

In [ ]:
# !pip install neo4j xgboost scikit-learn pandas matplotlib seaborn plotly openpyxl anthropic tqdm

In [ ]:
import os, warnings, json
from dotenv import load_dotenv
load_dotenv()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix,
    mean_squared_error, r2_score, roc_auc_score
)
import xgboost as xgb

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
SEED = 42
np.random.seed(SEED)

BASE = Path('.')
DIABETES_PATH   = BASE / 'diabetes_dataset.csv'
ACTIVITY_PATH   = BASE / 'activity_summary.xlsx'
WEARABLE_PATH   = BASE / 'Wearable_Dataset'
STRESS_V1_PATH  = BASE / 'Stress_Level_v1.csv'
STRESS_V2_PATH  = BASE / 'Stress_Level_v2.csv'

print('Paths configured.')
print(f'  Diabetes CSV   : {DIABETES_PATH.exists()}')
print(f'  Activity Excel : {ACTIVITY_PATH.exists()}')
print(f'  Wearable Data  : {WEARABLE_PATH.exists()} (PhysioNet required)')

---
## Section 1 — Diabetes Health Risk ML

In [ ]:
df = pd.read_csv(DIABETES_PATH)
print(f'Shape: {df.shape}')
print(f'\nDiabetes stage distribution:')
print(df['diabetes_stage'].value_counts())
print(f'\nDiagnosed diabetes: {df["diagnosed_diabetes"].value_counts().to_dict()}')
df.head(3)

In [ ]:
# EDA — key vitals distribution by diabetes stage
vitals = ['glucose_fasting', 'hba1c', 'bmi', 'heart_rate', 'systolic_bp']
stages = df['diabetes_stage'].unique()

fig, axes = plt.subplots(1, len(vitals), figsize=(20, 4))
for ax, col in zip(axes, vitals):
    for stage in stages:
        subset = df[df['diabetes_stage'] == stage][col].dropna()
        ax.hist(subset, alpha=0.5, bins=40, label=stage)
    ax.set_title(col.replace('_', ' ').title(), fontsize=10)
    ax.set_xlabel('')
axes[0].legend(fontsize=7)
plt.suptitle('Vital Distributions by Diabetes Stage', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — numeric features
num_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[num_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            linewidths=0.3, cbar_kws={'shrink': 0.6})
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Feature engineering
df_ml = df.copy()

# Encode categoricals
cat_cols = ['gender', 'ethnicity', 'education_level', 'income_level',
            'employment_status', 'smoking_status']
le_map = {}
for col in cat_cols:
    le = LabelEncoder()
    df_ml[col + '_enc'] = le.fit_transform(df_ml[col].astype(str))
    le_map[col] = le

# Encode diabetes_stage as ordinal target for regression
stage_order = ['No Diabetes', 'Pre-Diabetes', 'Type 2', 'Type 1']
stage_map = {s: i for i, s in enumerate(stage_order)}
df_ml['stage_ord'] = df_ml['diabetes_stage'].map(stage_map).fillna(0).astype(int)

FEATURES = [
    'age', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate',
    'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides',
    'glucose_fasting', 'glucose_postprandial', 'insulin_level', 'hba1c',
    'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day',
    'alcohol_consumption_per_week', 'screen_time_hours_per_day',
    'family_history_diabetes', 'hypertension_history', 'cardiovascular_history',
    'gender_enc', 'smoking_status_enc'
]

X = df_ml[FEATURES].fillna(df_ml[FEATURES].median())
y_cls = df_ml['diagnosed_diabetes']       # binary classification
y_reg = df_ml['diabetes_risk_score']      # risk score regression
y_stage = df_ml['stage_ord']              # multi-class stage

X_train, X_test, yc_train, yc_test, yr_train, yr_test, ys_train, ys_test = train_test_split(
    X, y_cls, y_reg, y_stage, test_size=0.2, random_state=SEED, stratify=y_cls
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

In [ ]:
# XGBoost binary classifier — diagnosed_diabetes
clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='logloss',
    random_state=SEED, n_jobs=-1
)
clf.fit(X_train, yc_train, eval_set=[(X_test, yc_test)], verbose=False)

yc_pred = clf.predict(X_test)
yc_prob = clf.predict_proba(X_test)[:, 1]

print('=== Diabetes Classifier ===')
print(classification_report(yc_test, yc_pred, target_names=['No Diabetes', 'Diagnosed']))
print(f'ROC-AUC: {roc_auc_score(yc_test, yc_prob):.4f}')

In [ ]:
# Feature importance
imp = pd.Series(clf.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
imp.head(15).plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.title('XGBoost — Top 15 Feature Importances (Diabetes Classifier)')
plt.tight_layout()
plt.show()

In [ ]:
# XGBoost regressor — diabetes_risk_score
reg = xgb.XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1
)
reg.fit(X_train, yr_train, eval_set=[(X_test, yr_test)], verbose=False)

yr_pred = reg.predict(X_test)
rmse = np.sqrt(mean_squared_error(yr_test, yr_pred))
r2   = r2_score(yr_test, yr_pred)
print(f'=== Risk Score Regressor === RMSE: {rmse:.3f}  R²: {r2:.4f}')

plt.figure(figsize=(6, 5))
plt.scatter(yr_test[:2000], yr_pred[:2000], alpha=0.2, s=5, color='tomato')
mn, mx = yr_test.min(), yr_test.max()
plt.plot([mn, mx], [mn, mx], 'k--', lw=1)
plt.xlabel('Actual Risk Score')
plt.ylabel('Predicted Risk Score')
plt.title(f'Risk Score: Actual vs Predicted  (R²={r2:.3f})')
plt.tight_layout()
plt.show()

In [ ]:
# K-Means patient archetypes
scaler = StandardScaler()
cluster_features = ['bmi', 'glucose_fasting', 'hba1c', 'systolic_bp',
                    'physical_activity_minutes_per_week', 'diabetes_risk_score']
X_clust = scaler.fit_transform(df_ml[cluster_features].fillna(df_ml[cluster_features].median()))

# Elbow method
inertias = []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(X_clust)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method — Patient Archetypes')
plt.tight_layout()
plt.show()

In [ ]:
# Fit final K-Means with k=5
K_FINAL = 5
kmeans = KMeans(n_clusters=K_FINAL, random_state=SEED, n_init=10)
df_ml['cluster'] = kmeans.fit_predict(X_clust)

cluster_summary = df_ml.groupby('cluster')[cluster_features + ['diagnosed_diabetes']].mean().round(2)
print('=== Patient Archetype Centroids ===')
display(cluster_summary)

# Scatter: BMI vs HbA1c colored by cluster
plt.figure(figsize=(8, 5))
for c in range(K_FINAL):
    sub = df_ml[df_ml['cluster'] == c]
    plt.scatter(sub['bmi'].sample(500, random_state=SEED),
                sub['hba1c'].sample(500, random_state=SEED),
                alpha=0.4, s=8, label=f'Cluster {c}')
plt.xlabel('BMI')
plt.ylabel('HbA1c')
plt.title('Patient Archetypes — BMI vs HbA1c')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Isolation Forest — anomalous patients
iso = IsolationForest(contamination=0.05, random_state=SEED, n_jobs=-1)
df_ml['anomaly'] = iso.fit_predict(X)
df_ml['anomaly_score'] = iso.decision_function(X)

anomalies = df_ml[df_ml['anomaly'] == -1]
print(f'Anomalies detected: {len(anomalies)} ({len(anomalies)/len(df_ml)*100:.1f}%)')
print('\nAnomaly vs Normal — mean vitals:')
display(df_ml.groupby('anomaly')[['glucose_fasting', 'hba1c', 'bmi', 'diabetes_risk_score']].mean().round(2))

---
## Section 2 — Device Benchmark Layer (activity_summary.xlsx)

In [ ]:
act = pd.read_excel(ACTIVITY_PATH)
print(f'Shape: {act.shape}')
print(f'Devices: {act["device"].unique()}')
print(f'Activities: {act["activity"].unique()}')
act.head(3)

In [ ]:
# Device signal quality comparison
device_summary = act.groupby('device')[[
    'heart_rate_mean', 'signal_quality_score', 'measurement_reliability',
    'anomaly_rate_pct', 'firmware_smoothing_factor'
]].mean().round(3).sort_values('signal_quality_score', ascending=False)

print('=== Device Signal Quality Ranking ===')
display(device_summary)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, color in zip(axes,
    ['signal_quality_score', 'measurement_reliability', 'anomaly_rate_pct'],
    ['steelblue', 'seagreen', 'tomato']):
    vals = device_summary[col]
    ax.barh(vals.index, vals.values, color=color, alpha=0.8)
    ax.set_title(col.replace('_', ' ').title(), fontsize=10)
    ax.invert_yaxis()
plt.suptitle('Wearable Device Benchmarks', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# HR by activity type across devices
activity_device = act[act['breakdown_type'] == 'activity_device'].copy()
pivot = activity_device.pivot_table(
    index='activity', columns='device', values='heart_rate_mean', aggfunc='mean'
)

plt.figure(figsize=(12, 6))
pivot.plot(kind='bar', ax=plt.gca(), alpha=0.8)
plt.title('Mean Heart Rate by Activity & Device')
plt.ylabel('HR (bpm)')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Device', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
## Section 3 — Neo4j Graph Layer

**Prerequisites:** Neo4j Desktop or Aura running.  
Set `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` as environment variables or edit the cell below.

In [ ]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
load_dotenv(override=True)

NEO4J_URI      = os.getenv('NEO4J_URI',      'bolt://localhost:7687')
NEO4J_USER     = os.getenv('NEO4J_USER',     'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print('Neo4j connected.')
    NEO4J_AVAILABLE = True
except Exception as e:
    print(f'Neo4j not available: {e}')
    print('Graph loading cells will be skipped.')
    NEO4J_AVAILABLE = False

In [ ]:
def run_query(cypher, params=None):
    """Run a Cypher query and return results as a list of dicts."""
    with driver.session() as session:
        result = session.run(cypher, params or {})
        return [dict(r) for r in result]

if NEO4J_AVAILABLE:
    # Schema: constraints + indexes
    schema_queries = [
        'CREATE CONSTRAINT IF NOT EXISTS FOR (p:Patient) REQUIRE p.patient_id IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (s:Subject) REQUIRE s.subject_id IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (ses:Session) REQUIRE ses.session_id IS UNIQUE',
        'CREATE INDEX IF NOT EXISTS FOR (p:Patient) ON (p.cluster)',
        'CREATE INDEX IF NOT EXISTS FOR (p:Patient) ON (p.diabetes_stage)',
        'CREATE INDEX IF NOT EXISTS FOR (tw:TimeWindow) ON (tw.timestamp)',
    ]
    for q in schema_queries:
        run_query(q)
    print('Schema ready.')

In [ ]:
if NEO4J_AVAILABLE:
    # Load Patient nodes in batches of 1000
    from tqdm import tqdm

    BATCH = 1000
    patient_cols = ['age', 'gender', 'bmi', 'systolic_bp', 'diastolic_bp', 'heart_rate',
                    'glucose_fasting', 'hba1c', 'diabetes_risk_score',
                    'diabetes_stage', 'diagnosed_diabetes', 'cluster', 'anomaly']

    load_cypher = """
    UNWIND $rows AS row
    MERGE (p:Patient {patient_id: row.patient_id})
    SET
        p.age               = row.age,
        p.gender            = row.gender,
        p.bmi               = row.bmi,
        p.systolic_bp       = row.systolic_bp,
        p.heart_rate        = row.heart_rate,
        p.glucose_fasting   = row.glucose_fasting,
        p.hba1c             = row.hba1c,
        p.risk_score        = row.diabetes_risk_score,
        p.diabetes_stage    = row.diabetes_stage,
        p.diagnosed         = row.diagnosed_diabetes,
        p.cluster           = row.cluster,
        p.is_anomaly        = (row.anomaly = -1)
    """

    df_ml['patient_id'] = ['P' + str(i).zfill(6) for i in range(len(df_ml))]
    rows = df_ml[['patient_id'] + patient_cols].to_dict('records')

    for i in tqdm(range(0, len(rows), BATCH), desc='Loading patients'):
        run_query(load_cypher, {'rows': rows[i:i+BATCH]})

    count = run_query('MATCH (p:Patient) RETURN count(p) AS n')[0]['n']
    print(f'Patient nodes in graph: {count}')

In [ ]:
if NEO4J_AVAILABLE:
    # SIMILAR_TO edges — connect patients within the same cluster (sample 200 per cluster)
    similar_cypher = """
    UNWIND $pairs AS pair
    MATCH (a:Patient {patient_id: pair.a})
    MATCH (b:Patient {patient_id: pair.b})
    MERGE (a)-[r:SIMILAR_TO]-(b)
    SET r.cluster = pair.cluster
    """

    pairs = []
    for c in range(K_FINAL):
        cluster_ids = df_ml[df_ml['cluster'] == c]['patient_id'].sample(
            min(100, (df_ml['cluster'] == c).sum()), random_state=SEED
        ).tolist()
        for i in range(len(cluster_ids)):
            for j in range(i+1, min(i+4, len(cluster_ids))):
                pairs.append({'a': cluster_ids[i], 'b': cluster_ids[j], 'cluster': c})

    for i in range(0, len(pairs), 500):
        run_query(similar_cypher, {'pairs': pairs[i:i+500]})

    print(f'SIMILAR_TO relationships created: {len(pairs)}')

---
## Section 4 — PhysioNet Wearable Layer

Download the dataset from [PhysioNet](https://physionet.org/content/wearable-exam-stress/1.0.0/)  
Place `Wearable_Dataset/`, `Stress_Level_v1.csv`, `Stress_Level_v2.csv` in the same folder as this notebook.  
If the folder is not present, a **mock dataset** is generated automatically for demo purposes.

In [ ]:
import datetime

def moving_average_acc(acc_data):
    """Empatica E4 accelerometer moving average (32-sample windows)."""
    avg, prevX, prevY, prevZ = 0, 0, 0, 0
    results = []
    for i in range(0, len(acc_data), 32):
        buf = acc_data[i:i+32]
        if len(buf) == 0:
            continue
        buffX, buffY, buffZ = buf[:, 0], buf[:, 1], buf[:, 2]
        s = sum(max(abs(buffX[j]-prevX), abs(buffY[j]-prevY), abs(buffZ[j]-prevZ))
                for j in range(len(buffX)))
        prevX, prevY, prevZ = buffX[-1], buffY[-1], buffZ[-1]
        avg = avg * 0.9 + (s / 32) * 0.1
        results.append(avg)
    return np.array(results)


def read_e4_signals(subject_folder: Path) -> dict:
    """Read all Empatica E4 CSV files for one subject."""
    signals, timelines, fs_map = {}, {}, {}
    desired = ['EDA.csv', 'BVP.csv', 'HR.csv', 'TEMP.csv', 'tags.csv', 'ACC.csv']

    # Get start UTC from EDA header
    eda_path = subject_folder / 'EDA.csv'
    utc_start = pd.read_csv(eda_path, nrows=0).columns[0]

    for fname in desired:
        fpath = subject_folder / fname
        if not fpath.exists():
            continue
        if fname == 'tags.csv':
            try:
                tags_df = pd.read_csv(fpath, header=None)
                tags_utc = np.insert(tags_df.values.flatten(), 0, utc_start)
                tags_sec = [(datetime.datetime.strptime(str(t), '%Y-%m-%d %H:%M:%S') -
                             datetime.datetime.strptime(str(tags_utc[0]), '%Y-%m-%d %H:%M:%S')
                             ).total_seconds() for t in tags_utc]
                signals['tags'] = tags_sec
            except pd.errors.EmptyDataError:
                signals['tags'] = []
        else:
            df_s = pd.read_csv(fpath)
            fs = int(df_s.iloc[0, 0])
            data = df_s.iloc[1:].values
            timeline = np.linspace(0, len(data)/fs, len(data))
            signal_name = fname.replace('.csv', '')
            signals[signal_name] = data
            timelines[signal_name] = timeline
            fs_map[signal_name] = fs
    return signals, timelines, fs_map


def load_physionet(dataset_path: Path) -> dict:
    """Load all subjects across all sessions."""
    all_data = {}
    for state in ['STRESS', 'AEROBIC', 'ANAEROBIC']:
        state_path = dataset_path / state
        if not state_path.exists():
            continue
        all_data[state] = {}
        for subject_folder in sorted(state_path.iterdir()):
            if subject_folder.is_dir():
                sig, tl, fs = read_e4_signals(subject_folder)
                all_data[state][subject_folder.name] = {'signals': sig, 'timelines': tl, 'fs': fs}
    return all_data


def generate_mock_wearable(n_subjects=10, session_secs=600) -> dict:
    """
    Generate mock E4-style physiological data for demo purposes.
    State-dependent realistic ranges:
      STRESS:   HR 85-110, EDA 5-20, TEMP 32-34
      AEROBIC:  HR 120-160, EDA 10-30, TEMP 34-37
      ANAEROBIC:HR 150-180, EDA 15-40, TEMP 35-38
    """
    rng = np.random.default_rng(SEED)
    state_params = {
        'STRESS':    dict(hr=(90, 12), eda=(10, 5), temp=(33, 0.5)),
        'AEROBIC':   dict(hr=(140, 15), eda=(18, 6), temp=(35.5, 0.7)),
        'ANAEROBIC': dict(hr=(165, 10), eda=(25, 8), temp=(36.5, 0.6)),
    }
    mock = {}
    for state, params in state_params.items():
        mock[state] = {}
        for i in range(1, n_subjects + 1):
            sid = f'MOCK_{state[0]}{i:02d}'
            t = np.arange(session_secs)
            hr   = rng.normal(*params['hr'],   size=session_secs).clip(40, 200)
            eda  = rng.normal(*params['eda'],  size=session_secs).clip(0, 60)
            temp = rng.normal(*params['temp'], size=session_secs).clip(28, 40)
            mock[state][sid] = {
                'timeline': t,
                'HR': hr, 'EDA': eda, 'TEMP': temp,
                'state': state
            }
    return mock


print('Wearable loader functions defined.')

In [ ]:
# Load real PhysioNet data if available, else use mock
if WEARABLE_PATH.exists():
    print('Loading PhysioNet data...')
    physionet_data = load_physionet(WEARABLE_PATH)
    USING_MOCK = False
    print(f'Loaded states: {list(physionet_data.keys())}')
    for state, subjects in physionet_data.items():
        print(f'  {state}: {len(subjects)} subjects')
else:
    print('PhysioNet Wearable_Dataset not found — using MOCK data for demo.')
    physionet_data = generate_mock_wearable(n_subjects=18)
    USING_MOCK = True
    print(f'Mock subjects per state: 18')

In [ ]:
def extract_window_features(hr_arr, eda_arr, temp_arr, window_sec=30, step_sec=10):
    """
    Slide a window over 1-Hz signals and extract statistical features.
    Returns a DataFrame with one row per window.
    """
    rows = []
    n = min(len(hr_arr), len(eda_arr), len(temp_arr))
    for start in range(0, n - window_sec, step_sec):
        end = start + window_sec
        hr_w  = hr_arr[start:end]
        eda_w = eda_arr[start:end]
        tmp_w = temp_arr[start:end]
        rows.append({
            'window_start': start,
            'hr_mean': np.mean(hr_w),   'hr_std': np.std(hr_w),   'hr_max': np.max(hr_w),
            'eda_mean': np.mean(eda_w), 'eda_std': np.std(eda_w), 'eda_max': np.max(eda_w),
            'temp_mean': np.mean(tmp_w),'temp_std': np.std(tmp_w),
            # RMSSD-like HR variability proxy
            'hr_rmssd': np.sqrt(np.mean(np.diff(hr_w)**2)) if len(hr_w) > 1 else 0,
        })
    return pd.DataFrame(rows)


# Build feature dataset for stress classifier
window_records = []

for state, subjects in physionet_data.items():
    for sid, data in subjects.items():
        if USING_MOCK:
            hr   = data['HR']
            eda  = data['EDA']
            temp = data['TEMP']
        else:
            sigs = data['signals']
            # PhysioNet signals are 2D arrays; flatten to 1D and resample to ~1Hz
            def _flatten_resample(arr, fs, target_fs=1):
                flat = arr.flatten() if arr.ndim > 1 else arr
                step = max(1, int(fs / target_fs))
                return flat[::step].astype(float)
            hr   = _flatten_resample(sigs.get('HR',   np.array([[75]])), data['fs'].get('HR', 1))
            eda  = _flatten_resample(sigs.get('EDA',  np.array([[1]])),  data['fs'].get('EDA', 4))
            temp = _flatten_resample(sigs.get('TEMP', np.array([[33]])), data['fs'].get('TEMP', 4))

        feat_df = extract_window_features(hr, eda, temp)
        feat_df['subject_id'] = sid
        feat_df['state'] = state
        window_records.append(feat_df)

wearable_df = pd.concat(window_records, ignore_index=True)
print(f'Wearable feature windows: {len(wearable_df)}')
wearable_df.head()

In [ ]:
# Stress Classifier: STRESS vs REST (AEROBIC + ANAEROBIC = exercise, not rest; use binary stress/non-stress)
wearable_df['is_stress'] = (wearable_df['state'] == 'STRESS').astype(int)

W_FEATURES = ['hr_mean', 'hr_std', 'hr_max', 'hr_rmssd',
               'eda_mean', 'eda_std', 'eda_max',
               'temp_mean', 'temp_std']

Xw = wearable_df[W_FEATURES].fillna(0)
yw = wearable_df['is_stress']

Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    Xw, yw, test_size=0.2, random_state=SEED, stratify=yw
)

stress_clf = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    use_label_encoder=False, eval_metric='logloss',
    random_state=SEED, n_jobs=-1
)
stress_clf.fit(Xw_train, yw_train, eval_set=[(Xw_test, yw_test)], verbose=False)

yw_pred = stress_clf.predict(Xw_test)
yw_prob = stress_clf.predict_proba(Xw_test)[:, 1]

print('=== Wearable Stress Classifier ===')
print(classification_report(yw_test, yw_pred, target_names=['Non-Stress', 'Stress']))
print(f'ROC-AUC: {roc_auc_score(yw_test, yw_prob):.4f}')

# Feature importance
w_imp = pd.Series(stress_clf.feature_importances_, index=W_FEATURES).sort_values(ascending=False)
w_imp.plot(kind='barh', figsize=(8, 4), color='coral', title='Stress Classifier Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Add stress predictions back to wearable_df
wearable_df['stress_prob'] = stress_clf.predict_proba(Xw)[:, 1]
wearable_df['stress_pred'] = stress_clf.predict(Xw)

# Visualize: mean stress probability per subject per state
subj_stress = wearable_df.groupby(['subject_id', 'state'])['stress_prob'].mean().reset_index()

plt.figure(figsize=(12, 5))
for state, color in [('STRESS', 'tomato'), ('AEROBIC', 'steelblue'), ('ANAEROBIC', 'seagreen')]:
    sub = subj_stress[subj_stress['state'] == state]
    plt.bar(sub['subject_id'], sub['stress_prob'], alpha=0.7, label=state, color=color)
plt.axhline(0.5, color='black', linestyle='--', lw=1)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.ylabel('Mean Stress Probability')
plt.title('Per-Subject Stress Probability by Session Type')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
if NEO4J_AVAILABLE:
    # Load Subject → Session → TimeWindow nodes
    subject_cypher = """
    UNWIND $subjects AS s
    MERGE (sub:Subject {subject_id: s.subject_id})
    SET sub.is_mock = s.is_mock
    MERGE (ses:Session {session_id: s.session_id})
    SET ses.state = s.state, ses.subject_id = s.subject_id
    MERGE (sub)-[:PARTICIPATED_IN]->(ses)
    """

    window_cypher = """
    UNWIND $rows AS r
    MATCH (ses:Session {session_id: r.session_id})
    MERGE (tw:TimeWindow {window_id: r.window_id})
    SET tw.window_start = r.window_start,
        tw.hr_mean      = r.hr_mean,
        tw.eda_mean     = r.eda_mean,
        tw.temp_mean    = r.temp_mean,
        tw.stress_prob  = r.stress_prob,
        tw.stress_pred  = r.stress_pred
    MERGE (ses)-[:HAS_WINDOW]->(tw)
    """

    # Build subject/session records
    subj_records, win_records = [], []
    for state, subjects in physionet_data.items():
        for sid in subjects:
            session_id = f'{sid}_{state}'
            subj_records.append({'subject_id': sid, 'session_id': session_id,
                                  'state': state, 'is_mock': USING_MOCK})

    # Build window records
    for _, row in wearable_df.iterrows():
        session_id = f"{row['subject_id']}_{row['state']}"
        window_id  = f"{session_id}_w{int(row['window_start'])}"
        win_records.append({
            'session_id':   session_id,
            'window_id':    window_id,
            'window_start': int(row['window_start']),
            'hr_mean':      round(float(row['hr_mean']), 2),
            'eda_mean':     round(float(row['eda_mean']), 2),
            'temp_mean':    round(float(row['temp_mean']), 2),
            'stress_prob':  round(float(row['stress_prob']), 4),
            'stress_pred':  int(row['stress_pred']),
        })

    run_query(subject_cypher, {'subjects': subj_records})
    for i in range(0, len(win_records), 500):
        run_query(window_cypher, {'rows': win_records[i:i+500]})

    print(f'Subjects loaded: {len(subj_records)}')
    print(f'Time windows loaded: {len(win_records)}')

---
## Section 5 — Graph Queries & Insights

In [ ]:
if NEO4J_AVAILABLE:
    # High-risk patients with anomalies
    q1 = """
    MATCH (p:Patient)
    WHERE p.risk_score > 70 AND p.is_anomaly = true
    RETURN p.patient_id, p.age, p.bmi, p.hba1c, p.glucose_fasting,
           p.diabetes_stage, p.risk_score
    ORDER BY p.risk_score DESC LIMIT 10
    """
    res = run_query(q1)
    print('=== High-Risk Anomalous Patients ===')
    display(pd.DataFrame(res))

In [ ]:
if NEO4J_AVAILABLE:
    # Cluster composition by diabetes stage
    q2 = """
    MATCH (p:Patient)
    RETURN p.cluster AS cluster, p.diabetes_stage AS stage, count(*) AS cnt
    ORDER BY cluster, cnt DESC
    """
    res2 = pd.DataFrame(run_query(q2))
    pivot2 = res2.pivot_table(index='cluster', columns='stage', values='cnt', fill_value=0)
    print('=== Cluster × Diabetes Stage ===')
    display(pivot2)

In [ ]:
if NEO4J_AVAILABLE:
    # Subjects with high stress windows
    q3 = """
    MATCH (sub:Subject)-[:PARTICIPATED_IN]->(ses:Session)-[:HAS_WINDOW]->(tw:TimeWindow)
    WHERE ses.state = 'STRESS'
    RETURN sub.subject_id AS subject,
           avg(tw.stress_prob) AS avg_stress_prob,
           max(tw.hr_mean) AS peak_hr,
           max(tw.eda_mean) AS peak_eda
    ORDER BY avg_stress_prob DESC LIMIT 10
    """
    res3 = run_query(q3)
    print('=== Top Stress-Flagged Subjects ===')
    display(pd.DataFrame(res3))

---
## Section 6 — LLM Health Recommendations (Claude API)

Set `ANTHROPIC_API_KEY` as an environment variable.

In [ ]:
from rocketride import RocketRideClient
import os

ROCKETRIDE_AVAILABLE = bool(os.getenv('ROCKETRIDE_APIKEY'))
if not ROCKETRIDE_AVAILABLE:
    print('ROCKETRIDE_APIKEY not set — LLM cells will print the prompt only.')
else:
    rr_client = RocketRideClient() # auto-reads ROCKETRIDE_APIKEY from .env
    await rr_client.connect()
    pipeline = {
        'components': [{'id': 'chat', 'provider': 'chat', 'config': {}}],
        'source': 'chat',
        'project_id': 'vitalgraph'
    }
    result = await rr_client.use(pipeline=pipeline)
    rocket_token = result['token']
    print('RocketRide client ready.')


In [ ]:
def build_health_prompt(patient_row: dict, wearable_summary: dict = None,
                        similar_patients: list = None) -> str:
    """
    Build a structured health recommendation prompt.
    patient_row:      dict of a Patient record
    wearable_summary: dict with avg_stress_prob, peak_hr, peak_eda (from graph query)
    similar_patients: list of similar patient dicts from SIMILAR_TO query
    """
    p = patient_row
    lines = [
        'You are a clinical health assistant. Based on the following patient data, '
        'provide a concise, actionable health recommendation. '
        'Do not diagnose — focus on lifestyle and monitoring suggestions.',
        '',
        '## Patient Profile',
        f'- Age: {p.get("age", "N/A")}  |  Gender: {p.get("gender", "N/A")}  |  BMI: {p.get("bmi", "N/A")}',
        f'- Fasting Glucose: {p.get("glucose_fasting", "N/A")} mg/dL  |  HbA1c: {p.get("hba1c", "N/A")}%',
        f'- Systolic BP: {p.get("systolic_bp", "N/A")} mmHg  |  Heart Rate: {p.get("heart_rate", "N/A")} bpm',
        f'- Diabetes Stage: {p.get("diabetes_stage", "N/A")}  |  Risk Score: {p.get("diabetes_risk_score", "N/A"):.1f}/100',
        f'- Anomaly Flagged: {p.get("anomaly", 1) == -1}  |  Cluster Archetype: {p.get("cluster", "N/A")}',
    ]

    if wearable_summary:
        lines += [
            '',
            '## Wearable Session Data',
            f'- Average Stress Probability: {wearable_summary.get("avg_stress_prob", "N/A"):.0%}',
            f'- Peak Heart Rate: {wearable_summary.get("peak_hr", "N/A"):.0f} bpm',
            f'- Peak EDA (Electrodermal Activity): {wearable_summary.get("peak_eda", "N/A"):.1f} µS',
        ]

    if similar_patients:
        avg_risk = np.mean([sp.get('risk_score', 0) for sp in similar_patients])
        lines += [
            '',
            f'## Similar Patients ({len(similar_patients)} in same cluster)',
            f'- Mean Risk Score in Cluster: {avg_risk:.1f}/100',
        ]

    lines += [
        '',
        '## Request',
        '1. Summarize the key risk factors.',
        '2. Give 3 specific, prioritized lifestyle recommendations.',
        '3. Suggest which vitals to monitor more closely.',
        '4. Note any urgent concerns based on the anomaly flag or stress data.',
    ]
    return '\n'.join(lines)


print('Prompt builder ready.')

In [ ]:
# Demo: build a recommendation for a high-risk anomalous patient
condition = (df_ml['anomaly'] == -1) & (df_ml['diabetes_risk_score'] > 70)
candidates = df_ml[condition]

if len(candidates) > 0:
    high_risk_patient = candidates.head(1).iloc[0].to_dict()
else:
    # Fallback to just highest risk score if no anomaly matches > 70
    high_risk_patient = df_ml.sort_values('diabetes_risk_score', ascending=False).head(1).iloc[0].to_dict()

# Sample wearable context from the stress session
stress_candidates = wearable_df[wearable_df['state'] == 'STRESS']
if len(stress_candidates) > 0:
    # Use hr_mean and eda_mean instead of stress_prob if it doesn't exist yet
    stress_subject = stress_candidates.groupby('subject_id').agg(
        avg_stress_prob=('is_stress', 'mean'), # Approximation for prob if not predicted
        peak_hr=('hr_mean', 'max'),
        peak_eda=('eda_mean', 'max')
    ).reset_index().head(1).iloc[0].to_dict()
else:
    stress_subject = None

prompt = build_health_prompt(high_risk_patient, wearable_summary=stress_subject)
print('=== PROMPT PREVIEW ===')
print(prompt)


In [ ]:
from rocketride.schema import Question
if ROCKETRIDE_AVAILABLE:
    question = Question()
    question.addQuestion(prompt)
    response = await rr_client.chat(token=rocket_token, question=question)
    recommendation = response['answers'][0] if ('answers' in response and response['answers']) else 'No response.'
    print('=== HEALTH RECOMMENDATION ===')
    print(recommendation)
else:
    print('Set ROCKETRIDE_APIKEY to get live recommendations.')


---
## Appendix — Save Models

In [ ]:
import pickle

models = {
    'diabetes_classifier': clf,
    'risk_regressor': reg,
    'kmeans': kmeans,
    'isolation_forest': iso,
    'stress_classifier': stress_clf,
    'scaler': scaler,
    'label_encoders': le_map,
    'features': FEATURES,
    'wearable_features': W_FEATURES,
}

with open('models.pkl', 'wb') as f:
    pickle.dump(models, f)

print('Models saved to models.pkl')